# Import library

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# Download and create dataloader

In [ ]:
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 32

class CIFAR10Pairs(torch.utils.data.Dataset):
    def __init__(self, train=True):
        self.data = datasets.CIFAR10(root='./data', train=train, download=True)

        self.transform_input = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Normalize((0.5,), (0.5,))
        ])

        self.transform_target = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img, _ = self.data[idx]
        input_img = self.transform_input(img)
        target_img = self.transform_target(img)

        return input_img, target_img


# Dataloader
dataset = CIFAR10Pairs()
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)


# Define GAN hyperparameters

In [ ]:
# hyperparameters
latent_dim = 100
num_classes = 10
embedding_dim = 100
lr = 2e-4
beta1 = 0.5
beta2 = 0.999
num_epochs = 10
lambda_l1 = 100

# Define generator

In [ ]:
class UNetGenerator(nn.Module):
    def __init__(self, in_channels=3, outchannel=3):
        super(UNetGenerator, self).__init__()

        def down_block(in_c, out_c, normalize=True):
            layers = [nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False)]
            if normalize:
                layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*layers)
        
        def up_block(in_c, out_c, dropout=0.0):
            layers = [
                nn.ConvTranspose2d(in_c, out_c, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True)
            ]
            if dropout:
                layers.append(nn.Dropout(dropout))
            return nn.Sequential(*layers)
        
        self.down1 = down_block(in_channels, 64, normalize=False)
        self.down2 = down_block(64, 128)
        self.down3 = down_block(128, 256)

        self.bottleneck = nn.Sequential(
            nn.Conv2d(256, 512, 4, 2, 1),
            nn.ReLU()
        )

        self.up3 = up_block(512, 256)
        self.up2 = up_block(512, 128) # skip connection from down2
        self.up1 = up_block(256, 64)  # skip connection from down1

        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, outchannel, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down2(d2)
        bn = self.bottleneck(d3)
        u3 = self.up3(bn)
        u2 = self.up2(torch.cat([u3, d3], dim=1))
        u1 = self.up1(torch.cat([u2, d2], dim=1))
        out = self.final(torch.cat([u1, d1], dim=1))
        return out

# Build discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, num_classes, img_size=32):
        super(Discriminator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, img_size*img_size)

        self.model = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=3, stride=2, padding=1), # 3 channels color + 1 label y
            nn.LeakyReLU(0.2),
            nn.Dropout(0.25),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ZeroPad2d((0, 1, 0, 1)),
            nn.BatchNorm2d(64, momentum=0.82),
            nn.LeakyReLU(0.25),
            nn.Dropout(0.25),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128, momentum=0.82),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.25),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256, momentum=0.8),
            nn.LeakyReLU(0.25),
            nn.Dropout(0.25),
            nn.Conv2d(256, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x, labels):
        label_input = self.label_emb(labels).view(labels.size(0), 1, 32, 32)
        x = torch.cat((x, label_input), dim=1)
        validity = self.model(x)
        return validity

# Initialize GAN component

In [ ]:
generator = UNetGenerator(latent_dim, num_classes, embedding_dim).to(device)

discriminator = Discriminator(num_classes).to(device)

# Loss function
adversarial_loss = nn.BCELoss()
l1_loss = nn.L1Loss()

# Optimizer
optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, beta2))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, beta2))

# Train

In [ ]:
# training loop
for epoch in range(num_epochs):
    
    for i, (input_img, target_img) in enumerate(dataloader):
        input_img = input_img.to(device)
        target_img = target_img.to(device)

        valid = torch.zeros_like(input_img.size(0), 1, 4, 4, device=device)
        fake = torch.zeros_like(valid)

        """ Train generator """
        optimizer_G.zero_grad()
        fake_img = generator(input_img)
        pred_fake = discriminator(fake_img, input_img)
        g_adv = adversarial_loss(pred_fake, valid)
        g_l1 = l1_loss(fake_img, target_img)
        g_loss = g_adv + lambda_l1*g_l1
        g_loss.backward()
        optimizer_G.step()

        """ Train discriminator """
        optimizer_D.zero_grad()
        pred_real = discriminator(target_img, input_img)
        loss_real = adversarial_loss(pred_real, valid)

        pred_fake = discriminator(fake_img.detach(), input_img)
        loss_fake = adversarial_loss(pred_fake, fake)
        
        d_loss = 0.5 * (loss_real, loss_fake)
        d_loss.bacward()
        optimizer_D.step()
        

        if (i + 1) % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{num_epochs}] Batch {i+1}/{len(dataloader)} "
                f"D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}"
            )
    
    # visualize generation
    if (epoch + 1) % 10 == 0:
        generator.eval()
        with torch.no_grad():
            z = torch.randn(16, latent_dim, device=device)
            label_sample = torch.tensor([i % 10 for i in range(16)], device=device)
            gen_imgs = generator(z, label_sample).cpu()
            grid = torchvision.utils.make_grid(gen_imgs, nrow=4, normalize=True)
            plt.imshow(np.transpose(grid, (1, 2, 0)))
            plt.axis("off")
            plt.title("Epoch {}".format(epoch+1))
            plt.show()
        generator.train()